In [ ]:
from OceanDataStore import OceanDataCatalog
from nemo_cookbook import NEMODataTree
import gsw
import xarray as xr
import os 
import dask 
from dask.distributed import Client, LocalCluster

import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt
import pandas as pd

In [228]:
dask.config.set({'temporary_directory': f"{os.getcwd()}/dask_tmp",
                 'local_directory': f"{os.getcwd()}/dask_tmp"
                 })

# Create Local Cluster:
cluster = LocalCluster(n_workers=4, threads_per_worker=3, memory_limit='5GB')
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 12,Total memory: 18.63 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36525,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:44401,Total threads: 3
Dashboard: http://127.0.0.1:37317/status,Memory: 4.66 GiB
Nanny: tcp://127.0.0.1:44451,


2026-10-01 22:09:30,538 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:46305' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('getitem-37a74d849e83cf60947118168b052ec1', 470, 0.09999999999999998, 0, 0), ('mean_combine-partial-7d213835b7ae6c15e38ddb7b4c829271', 119, 1, 0, 0), ('getitem-37a74d849e83cf60947118168b052ec1', 470, 0, 0, 0), ('mean_combine-partial-7d213835b7ae6c15e38ddb7b4c829271', 119, 0, 0, 0), ('mean_combine-partial-7d213835b7ae6c15e38ddb7b4c829271', 119, 2, 0, 0), ('getitem-37a74d849e83cf60947118168b052ec1', 473, 1.9, 0, 0)} (stimulus_id='handle-worker-cleanup-1790888970.5383983')
2026-10-01 22:09:30,546 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:40925' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('sigma0-492a02b382f3d56dcae8a402d57825d3', 472, 0, 0, 0)} (stimulus_id='handle-worker-cleanup-1790888970.5460691')
2026-10-01 22:09:30,569 - di

In [233]:
catalog = OceanDataCatalog(catalog_name="noc-stac") 

ds_domain = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/domain_cfg", consolidated=True, chunks={})

ds_gridT = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/T1m", consolidated=True, chunks={})
ds_gridT['sigma0'] = gsw.density.sigma0(CT=ds_gridT['thetao_con'], SA=ds_gridT['so_abs'])
ds_gridT['sigma0'].name = 'sigma0'

ds_gridU = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/U1m", consolidated=True, chunks={})
ds_gridV = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/V1m", consolidated=True, chunks={})

datasets = {"parent": {"domain": ds_domain, "gridT": ds_gridT, "gridU": ds_gridU, "gridV": ds_gridV}}

# Initialise a new NEMODataTree whose parent domain is zonally periodic & north-folding on F-points:
nemo = NEMODataTree.from_datasets(datasets=datasets, iperio=True, nftype="F", read_mask=True)

In [248]:
###
# overturnign streamfunction #
###
atlmask = ds_domain['atlmsk'].rename({"x":"i", "y":"j"}).astype(bool) # assign to V-grid
atlmask['i'] = atlmask['i'] + 1
atlmask['j'] = atlmask['j'] + 1.5
Psi = nemo['gridV/vo'].integral(dims=["i", "k"], cum_dims=["k"], dir="+1", mask=atlmask)
print('Overturning streamfunciton calculated')

###
# geostorphic compoennt 
###
f_V = ds_domain["ff_f"].rename({"x": "i", "y": "j"}) # corilis parameter on V-grid
f_V = f_V.assign_coords(i=nemo["gridV/vo"].i, j=nemo["gridV/vo"].j)

g = 9.8 # gravity 
rho_0 = 1026 # reference sea water denisty

dsigma_dx = nemo["gridT/sigma0"].derivative(dim="i").interp_to(to="V")
dv_dz = -g * dsigma_dx / (rho_0 * f_V)
v_g = dv_dz.integral(dims=["k"], cum_dims=["k"], dir="+1")
v_g_rel = v_g - v_g.isel(k=-1) # zero velocity at depth 

Psi_geostrophic = ((v_g_rel * nemo["gridV/e1v"]).sum(dim="i"))
Psi_geostrophic = Psi_geostrophic.where(atlmask.any(dim="i"))
print('Geostrophic streamfunciton calculated')

###
# ekamn compoennt 
###
f_U = ds_domain["ff_f"].rename({"x": "i", "y": "j"})
f_U = f_U.assign_coords(i=nemo["gridU/tauuo"].i, j=nemo["gridU/tauuo"].j)
M_e = nemo["gridU/tauuo"] / (rho_0 * f_U)
Psi_ek = (M_e * nemo["gridU/e1u"]).interp_to(to="V").sum(dim="i")
print('Ekman streamfunciton calculated')

###
# residaul 
###
Psi_residual = Psi - Psi_geostrophic - Psi_ek
print('Residual streamfunciton calculated')

Overturning streamfunciton calculated
Geostrophic streamfunciton calculated
Ekman streamfunciton calculated
Residual streamfunciton calculated


In [ ]:
# Psi_max = Psi.max(dim=["k", "j"]).compute()
# Psi_g_max = Psi_geostrophic.max(dim=["k", "j"])
# Psi_ek_max = Psi_ek.max(dim=["k", "j"])
# Psi_res_max = Psi_residual.max(dim=["k", "j"])

# fig, ax = plt.subplots(figsize=(12, 5))

# ax.plot(Psi.time_counter, Psi_max, label="Total")
# ax.plot(Psi_geostrophic.time_counter, Psi_g_max, label="Geostrophic")
# ax.plot(Psi_ek.time_counter, Psi_ek_max, label="Ekman")
# ax.plot(Psi_residual.time_counter, Psi_res_max, label="Residual")

# ax.set_xlabel("Time")
# ax.set_ylabel(r"Maximum $|\Psi|$ (m$^3$ s$^{-1}$)")
# ax.set_title("Maximum Atlantic overturning strength")
# ax.legend()
# ax.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

Task exception was never retrieved
future: <Task finished name='Task-171334' coro=<Client._gather.<locals>.wait() done, defined at /home/teaching/StreamfunctionReconstruction/venv/venv/lib/python3.12/site-packages/distributed/client.py:2399> exception=AllExit()>
Traceback (most recent call last):
  File "/home/teaching/StreamfunctionReconstruction/venv/venv/lib/python3.12/site-packages/distributed/client.py", line 2408, in wait
    raise AllExit()
distributed.client.AllExit
Task exception was never retrieved
future: <Task finished name='Task-171333' coro=<Client._gather.<locals>.wait() done, defined at /home/teaching/StreamfunctionReconstruction/venv/venv/lib/python3.12/site-packages/distributed/client.py:2399> exception=AllExit()>
Traceback (most recent call last):
  File "/home/teaching/StreamfunctionReconstruction/venv/venv/lib/python3.12/site-packages/distributed/client.py", line 2408, in wait
    raise AllExit()
distributed.client.AllExit


KeyboardInterrupt: 

1. Normal modes
2. Normal modes with surface modes 
3. PCA from information at boundaries (vs across the entire domain)
4. ICA? from information at boundaries (vs across the entire domain)

Comapre with the geostorphic streamfunction and the full overturning streamfunction.

In [252]:
atlmask_T = (ds_domain["atlmsk"].rename({"x": "i", "y": "j"}).astype(bool))
atlmask_T = atlmask_T.assign_coords(i=nemo["gridT/sigma0"].i,j=nemo["gridT/sigma0"].j)

sigma0_mean = (nemo["gridT/sigma0"].where(atlmask_T).mean( dim=["time_counter", "j", "i"], skipna=True))

N2_mean = (-g / rho_0 * sigma0_mean.differentiate("deptht"))
print("Mean N2 calculated")

N2_mean.to_netcdf("N2_mean.nc", compute=True)

Mean N2 calculated


Task exception was never retrieved
future: <Task finished name='Task-182955' coro=<Client._gather.<locals>.wait() done, defined at /home/teaching/StreamfunctionReconstruction/venv/venv/lib/python3.12/site-packages/distributed/client.py:2399> exception=AllExit()>
Traceback (most recent call last):
  File "/home/teaching/StreamfunctionReconstruction/venv/venv/lib/python3.12/site-packages/distributed/client.py", line 2408, in wait
    raise AllExit()
distributed.client.AllExit


KeyboardInterrupt: 